# Recipient names

Create a name-mapping worksheet, then apply a completed semicolon-separated mapping to the corpus.

In [ ]:
from shared import paths
from IPython.display import display


## Mapping worksheet

In [ ]:
import pandas as pd
from shared.utils import load_pickle

letters = load_pickle()
recipient_names = [letter.recipient for letter in letters if letter.recipient]
unique_recipients = list(dict.fromkeys(recipient_names))


def invert_name(name):
    tokens = name.strip().split()
    if not tokens:
        return ''
    last = tokens[-1]
    rest = ' '.join(tokens[:-1])
    return f'{last}, {rest}' if rest else last
inverted_names = [invert_name(name) for name in unique_recipients]
df = pd.DataFrame(
    {
        'raw_name': unique_recipients,
        'inverted_name': inverted_names,
        'normalized_name': [''] * len(unique_recipients),
    },
)
df.to_csv(
    paths.DATA_DIR / 'recipient_normalization_template.csv',
    sep=';',
    index=False,
    encoding='utf-8',
)


## Apply the mapping

In [ ]:
import pickle
import pandas as pd

path = str(paths.CORPUS_PICKLE)
csv_path = str(paths.RECIPIENT_MAPPING)
with open(path, 'rb') as f:
    letters = pickle.load(f)
df = pd.read_csv(csv_path, sep=';')
df.columns = df.columns.str.strip()
df['raw_name'] = df['raw_name'].str.strip()
df['normalized_name'] = df['normalized_name'].str.strip()
name_map = dict(zip(df['raw_name'], df['normalized_name']))
total = 0
mapped = 0
examples = []
for letter in letters:
    raw = letter.recipient.strip() if letter.recipient else None
    if raw:
        total += 1
        norm = name_map.get(raw, None)
        letter.norm_recipient = norm
        if norm:
            mapped += 1
            if len(examples) < 10:
                examples.append((raw, norm))
    else:
        letter.norm_recipient = None
print(' Recipient normalisation:')
print(f'Non-empty recipients: {total}')
print(f'Successfully normalised: {mapped}')
print(f'Share: {mapped / total:.1%}')
print('\nExamples:')
for raw, norm in examples:
    print(f'- {raw} → {norm}')
with open(path, 'wb') as f:
    pickle.dump(letters, f)
